# DS5 — Stochastic Processes (StoP) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**  
**Dataset:** DS5 (German Adolescents RCT, N=573)  
**Model family:** Stochastic Differential Equations (StoP)  
**Date:** 2026-04-16  

---

## Scope and purpose

This notebook implements the first standalone CAS model pass for DS5 under **Stage 4, Step 1** scope:

- **Goal:** Establish a working StoP pipeline on DS5; generate interpretable parameter estimates; compare across RCT conditions
- **Not goal:** Phenotype classification, personalization, cross-dataset application, or full validation
- **Output:** Per-participant SDE parameter estimates (mu_hat, sig_hat) + condition-level comparison + preliminary psychosocial correlates

### FLAG decisions in effect

| Flag | Decision |
|---|---|
| FLAG 1 | Simple constant-drift SDE (not OU/mean-reverting) |
| FLAG 2 | Simple method-of-moments estimators (mean/std of delta_X_n) |
| FLAG 3 | Option A — friction as drift reduction: h_c = mu_bar_control minus mu_bar_c |
| FLAG 4 | StM Boltzmann bridge noted as forward connection; NOT implemented here |

### WARNING — proxy status of the state variable

**The state variable X_t in this notebook is a behavioral proxy, not a directly observed latent psychological state.**

X_t = daily friction override rate = n_opened_day / n_friction_events_day

This is a count-based behavioral measurement. It approximates behavioral override tendency on a given day but does NOT directly measure urge, self-control capacity, habit strength, or any latent cognitive/psychological state. All SDE parameters inherit this proxy limitation. The mean-increment drift telescopes to an endpoint statistic, so an all-days OLS slope is reported as a robustness estimate. Raw diffusion also contains the analytic binomial sampling floor of a daily proportion, so behavioral variability is assessed only through the observed-to-floor ratio and excess component.

This distinction must be maintained throughout interpretation.

---

## Section 1: Pure Theory — Stochastic Processes in CAS

### What StoP models capture (paper section 4.1.3)

A Stochastic Differential Equation (SDE) models a system that evolves under both **deterministic drift** (a systematic push in one direction) and **random diffusion** (irreducible noise). In a CAS framing, SDE models are appropriate when:

- The system has a continuous state that evolves over time
- There is a discernible trend direction (drift) but also genuine randomness in each step
- The random component is not just measurement error but reflects real-world volatility (mood, context, competing demands)

### The core SDE (paper Eq. 17)

    dX_t = mu * dt + sigma * dW_t

where:
- X_t = system state at time t (here: behavioral override tendency, proxied by daily override rate)
- mu = drift coefficient — the deterministic rate of change per unit time
- sigma = diffusion coefficient — the amplitude of stochastic fluctuations
- W_t = standard Wiener process (Brownian motion)

### Discrete-time approximation: Euler-Maruyama (paper Eq. 19)

With time step delta_t = 1 day:

    X_{n+1} = X_n + mu + sigma * Z_n,   Z_n ~ N(0,1) i.i.d.

Values are clipped to [0,1] to respect the behavioral bounds of the override rate.

### The sigma/|mu| impulsivity ratio (paper section 4.1.3)

The ratio sigma/|mu| characterizes the relative dominance of noise vs drift:

- **Low ratio (drift-dominated):** Behavior changes in a consistent direction — systematic habituation or escalation
- **High ratio (noise-dominated):** Day-to-day variability swamps any directional trend — behavior appears erratic or context-sensitive

This ratio is a key individual-difference quantity for later phenotyping (Stage 4, Step 2).

### Friction as drift reduction (FLAG 3 — Option A)

For each condition c:

    h_c = mu_bar_control - mu_bar_c

where mu_bar_c is the mean estimated drift across participants in condition c.
A positive h_c means friction reduced the drift toward opening.
By definition: h_control = 0.

### Unification connections (informational only — not implemented here)

- **StoP -> DyS:** When sigma -> 0, the SDE reduces to the ODE dX/dt = mu. DyS (notebook 42) tests this limit.
- **StoP -> InT:** The stationary distribution P_inf(x) has entropy H(P_inf) connecting to information-theoretic behavioral uncertainty (notebook 45).
- **StoP -> StM (FLAG 4, deferred):** For a potential-well SDE, P_inf(x) proportional to exp(-2V(x)/sigma^2) — the Boltzmann connection. Deferred to later.

---

## Section 2: Final Equations Under FLAG Decisions

### State variable

    X_t^(i) = n_opened_day^(i)_t / n_friction_events_day^(i)_t   in [0, 1]

Participant i, calendar day t. Denominator = friction-decision events with outcome in {opened, dismissed}; `closed` is a later session-termination record and is excluded.

### Parameter estimators (FLAG 2 — method of moments)

    delta_X_n^(i) = X_{n+1}^(i) - X_n^(i)

    mu_hat^(i)  = mean(delta_X_n^(i))
    sig_hat^(i) = std(delta_X_n^(i), ddof=1)

Minimum threshold: T_i >= 7 days (at least 6 increments).

### Friction effect estimate (FLAG 3 — Option A)

    h_c = mu_bar_control - mu_bar_c   for c in {control, default, reintervention}

### Theory-to-data mapping

| Theoretical quantity | Data column / computation | Source file | Proxy limitation |
|---|---|---|---|
| State variable X_t | n_opened_day / n_events_day | ds5_behavioral_clean.csv | Override rate, not latent urge or self-control |
| Time index t | event_date (calendar day) | ds5_behavioral_clean.csv | Calendar days; ignores intra-day structure |
| Drift mu | mean(delta_X_n) | Computed from daily time series | Direction of change in override rate; not motivation |
| Diffusion sigma | std(delta_X_n, ddof=1) | Computed from daily time series | Includes context noise + measurement error |
| Experimental condition | study_condition | ds5_survey_features.csv (joined) | RCT assignment; individual compliance not tracked |
| Friction effect h_c | mu_bar_ctrl - mu_bar_c | Computed from condition means | Population-level; individual heterogeneity not captured |
| Psychosocial moderators | fomo_mean_pre, belong_mean_pre, ostracism_mean_pre | ds5_survey_features.csv | Pre-intervention only |

### Equation gaps (resolved vs deferred)

| Gap | Resolution |
|---|---|
| Continuous SDE to discrete daily X_t | Euler-Maruyama with delta_t=1 (FLAG 1) |
| MLE vs method of moments | Method of moments (FLAG 2) |
| Friction parametrization | Drift reduction only (FLAG 3 Option A) |
| Boundary behavior | Hard clip to [0,1] in simulation |
| Stationarity | Not formally tested; acknowledged as limitation |
| Boltzmann connection | Deferred (FLAG 4) |

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT    = Path('..').resolve()
CLEANED = ROOT / 'outputs' / 'cleaned'
FEAT    = ROOT / 'outputs' / 'features' / 'ds5'
REPORTS = ROOT / 'outputs' / 'model_outputs' / 'stochastic_processes'
FIGURES = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

RNG_SEED  = 42
MIN_DAYS  = 7    # minimum distinct observation days for SDE estimation
N_TRAJ    = 200  # Euler-Maruyama trajectories per representative participant

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('ROOT    :', ROOT)
print('CLEANED :', CLEANED)
print('FEAT    :', FEAT)
print('REPORTS :', REPORTS)
print('FIGURES :', FIGURES)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load behavioral data and join study_condition
# =============================================================================
# NOTE: study_condition is NOT in ds5_behavioral_clean.csv.
# Must join from ds5_survey_features.csv. See modeling_cautions.md section 6.

beh = pd.read_csv(CLEANED / 'ds5_behavioral_clean.csv', low_memory=False)

# FIX: use .dt.normalize() instead of .dt.date to keep event_date as a
# pandas Timestamp (not Python datetime.date). Python datetime.date objects
# are stored as object dtype; NaT becomes float NaN which cannot be compared
# to datetime.date in .min()/.max() -- causes TypeError on pandas >=2.0.
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)
beh['event_date'] = beh['event_timestamp_utc'].dt.normalize()   # UTC midnight Timestamps

print(f'Behavioral rows     : {len(beh):,}')
print(f'Participants        : {beh["participant_id_canonical"].nunique():,}')
n_nat = beh['event_date'].isna().sum()
if n_nat > 0:
    print(f'WARNING: {n_nat} rows with NaT event_date (dropped downstream)')
print(f'Date range          : {beh["event_date"].dropna().min().date()} -> {beh["event_date"].dropna().max().date()}')
print(f'\nOutcome counts:')
print(beh['event_outcome_simple'].value_counts())

surv = pd.read_csv(FEAT / 'ds5_survey_features.csv')
print(f'\nSurvey rows         : {len(surv):,}')
print(surv['study_condition'].value_counts())

beh = beh.merge(
    surv[['participant_id_canonical', 'study_condition']],
    on='participant_id_canonical', how='left'
)
n_no_cond = beh['study_condition'].isna().sum()
print(f'\nEvents with NaN condition: {n_no_cond:,}')
print('(Expected: ~1 participant has no pre-survey record)')

In [ ]:
# =============================================================================
# Filter to friction-relevant events
# =============================================================================
# Retain friction-decision outcomes {opened, dismissed}; session-termination and non-decision rows are excluded.
# NOTE: DS5 has is_intervention_event=0 for ALL events -- do not use for filtering.

VALID_OUTCOMES = {'opened', 'dismissed'}
beh_fric = beh[beh['event_outcome_simple'].isin(VALID_OUTCOMES)].copy()
beh_fric['opened_flag'] = (beh_fric['event_outcome_simple'] == 'opened').astype(int)

print(f'Events after filter : {len(beh_fric):,}')
print(f'Dropped             : {len(beh) - len(beh_fric):,}')

# STRUCTURAL NOTE: DS5 control group is NOT friction-free.
# Control participants produce both opened and dismissed decision outcomes.
# The control-arm decision rate is recomputed below from opened plus dismissed rows only.
# X_t is well-defined for all three conditions.
ctrl_out = beh_fric[beh_fric['study_condition']=='control']['event_outcome_simple'].value_counts(normalize=True)
print(f'\nControl outcome proportions (confirms control not friction-free):')
print(ctrl_out.round(3))

In [ ]:
# =============================================================================
# Compute daily X_t per participant
# =============================================================================
# X_t^(i) = n_opened_day / n_friction_events_day  in [0,1]
# PROXY WARNING: count-based behavioral measurement, not a latent state.

daily = (
    beh_fric
    .groupby(['participant_id_canonical', 'study_condition', 'event_date'])
    .agg(n_opened=('opened_flag','sum'), n_events=('opened_flag','count'))
    .reset_index()
)
daily['X_t'] = daily['n_opened'] / daily['n_events']
daily = daily.sort_values(['participant_id_canonical','event_date']).reset_index(drop=True)

print(f'Daily records  : {len(daily):,}')
print(f'Participants   : {daily["participant_id_canonical"].nunique():,}')
print(f'X_t range      : [{daily["X_t"].min():.3f}, {daily["X_t"].max():.3f}]')
print(f'X_t mean       : {daily["X_t"].mean():.3f}')
print(f'\nDays per participant:')
print(daily.groupby('participant_id_canonical')['event_date'].nunique().describe().round(1))

In [ ]:
# =============================================================================
# Apply minimum data threshold (MIN_DAYS = 7)
# =============================================================================

n_days_pp  = daily.groupby('participant_id_canonical')['event_date'].nunique().rename('n_days')
eligible   = n_days_pp[n_days_pp >= MIN_DAYS].index
daily_elig = daily[daily['participant_id_canonical'].isin(eligible)].copy()

n_total = daily['participant_id_canonical'].nunique()
n_elig  = len(eligible)
print(f'Total participants      : {n_total}')
print(f'Eligible (>={MIN_DAYS} days)  : {n_elig} ({100*n_elig/n_total:.1f}%)')
print(f'Excluded (<{MIN_DAYS} days)   : {n_total-n_elig} ({100*(n_total-n_elig)/n_total:.1f}%)')
print(f'\nEligible by condition:')
print(daily_elig.groupby('study_condition')['participant_id_canonical'].nunique())

---

## Section 3: Parameter Estimation

Estimating drift mu_hat and diffusion sig_hat per participant (method of moments, FLAG 2).

In [ ]:
# =============================================================================
# SDE parameter estimation: method of moments
# =============================================================================

def estimate_sde_params(x_series):
    """
    Estimate constant-drift SDE parameters via method of moments.

    mu_hat  = mean(delta_X_n)          estimated drift
    sig_hat = std(delta_X_n, ddof=1)   estimated diffusion

    PROXY NOTE: parameters describe behavioral proxy X_t, not latent psychology.
    """
    x  = np.asarray(x_series, dtype=float)
    dx = np.diff(x)
    if len(dx) < 2:
        return np.nan, np.nan, len(dx)
    return float(np.mean(dx)), float(np.std(dx, ddof=1)), len(dx)


records = []
for pid, grp in daily_elig.groupby('participant_id_canonical'):
    g        = grp.sort_values('event_date')
    cond     = g['study_condition'].iloc[0]
    x_series = g['X_t'].values
    n_series = g['n_events'].to_numpy(dtype=float)
    mu_hat, sig_hat, n_incr = estimate_sde_params(x_series)

    # Robustness quantities required by the observation model. mu_hat is the
    # endpoint slope because mean(diff(X)) telescopes; mu_ols uses every day.
    t_days = (g['event_date'] - g['event_date'].min()).dt.days.to_numpy(dtype=float)
    ols = stats.linregress(t_days, x_series) if np.unique(t_days).size >= 2 else None
    mu_ols = float(ols.slope) if ols is not None else np.nan
    mu_ols_se = float(ols.stderr) if ols is not None else np.nan

    # X_t is a binomial proportion. This is the analytic sampling SD of adjacent
    # differences under constant p, evaluated at the participant mean.
    p_bar = float(np.mean(x_series))
    floor_terms = p_bar * (1.0 - p_bar) * (1.0 / n_series[:-1] + 1.0 / n_series[1:])
    sig_floor = float(np.sqrt(np.mean(floor_terms))) if len(floor_terms) else np.nan
    sig_floor_ratio = sig_hat / sig_floor if np.isfinite(sig_floor) and sig_floor > 0 else np.nan
    sig_excess = float(np.sqrt(max(sig_hat**2 - sig_floor**2, 0.0))) if np.isfinite(sig_floor) else np.nan
    n_high = int((x_series > 0.5).sum())
    p_high_days = n_high / len(x_series)
    p_high_days_smoothed = (n_high + 1.0) / (len(x_series) + 2.0)
    records.append(dict(
        participant_id_canonical=pid,
        study_condition=cond,
        n_days=len(x_series),
        n_increments=n_incr,
        x_mean=p_bar,
        mean_events_per_day=float(np.mean(n_series)),
        p_high_days=p_high_days,
        p_high_days_smoothed=p_high_days_smoothed,
        mu_hat=mu_hat,
        mu_ols=mu_ols,
        mu_ols_se=mu_ols_se,
        sig_hat=sig_hat,
        sig_floor=sig_floor,
        sig_floor_ratio=sig_floor_ratio,
        sig_excess=sig_excess,
    ))

params = pd.DataFrame(records)
print(f'Parameter records : {len(params):,}')
print(f'NaN mu_hat        : {params["mu_hat"].isna().sum()}')
print(f'NaN sig_hat       : {params["sig_hat"].isna().sum()}')
print(f'\nSummary:')
print(params[['mu_hat','mu_ols','sig_hat','sig_floor','sig_floor_ratio','sig_excess','n_days']].describe().round(4))

In [ ]:
# =============================================================================
# Sanity check: x_mean vs rate_opened from Stage 1 features
# =============================================================================
# Small differences expected (daily-average vs event-average weighting).
# Differences > 0.20 warrant investigation.

beh_feat = pd.read_csv(FEAT / 'ds5_behavior_features.csv')
check    = params.merge(beh_feat[['participant_id_canonical','rate_opened']],
                        on='participant_id_canonical', how='left')
diff     = (check['x_mean'] - check['rate_opened']).abs()
print('|x_mean - rate_opened|:')
print(diff.describe().round(4))
print(f'Participants with |diff| > 0.20 : {(diff > 0.20).sum()}')

In [ ]:
# =============================================================================
# sigma/|mu| impulsivity ratio
# =============================================================================
# Participants with |mu_hat| < MU_THRESHOLD get NaN (avoids division by near-zero).

MU_THRESHOLD = 0.005
params['sigma_mu_ratio'] = np.where(
    params['mu_hat'].abs() < MU_THRESHOLD,
    np.nan,
    params['sig_hat'] / params['mu_hat'].abs()
)

n_nan = params['sigma_mu_ratio'].isna().sum()
print(f'sigma/|mu| ratio (excluding |mu| < {MU_THRESHOLD}):')
print(params['sigma_mu_ratio'].describe().round(2))
print(f'\nNaN (|mu| < {MU_THRESHOLD}): {n_nan} ({100*n_nan/len(params):.1f}%)')
print('\nMedian ratio by condition:')
print(params.groupby('study_condition')['sigma_mu_ratio'].median().round(2))

---

## Section 4: Condition Comparison and Friction Effect h_c

In [ ]:
# =============================================================================
# Condition-level statistics and h_c (FLAG 3 Option A)
# =============================================================================

cond_stats = (
    params.groupby('study_condition')
    .agg(n=('mu_hat','count'), mu_mean=('mu_hat','mean'), mu_median=('mu_hat','median'),
         mu_std=('mu_hat','std'), mu_ols_mean=('mu_ols','mean'),
         sig_mean=('sig_hat','mean'), sig_median=('sig_hat','median'), sig_std=('sig_hat','std'),
         sig_floor_mean=('sig_floor','mean'), sig_floor_ratio_mean=('sig_floor_ratio','mean'),
         sig_excess_mean=('sig_excess','mean'), x_mean_avg=('x_mean','mean'),
         mean_events_per_day=('mean_events_per_day','mean'))
    .reset_index()
)

mu_bar_ctrl = cond_stats.loc[cond_stats['study_condition']=='control','mu_mean'].values[0]
cond_stats['h_drift_reduction'] = mu_bar_ctrl - cond_stats['mu_mean']

print('Condition-level SDE summary:')
print(cond_stats[['study_condition','n','mu_mean','mu_median','mu_std',
                   'mu_ols_mean','sig_mean','sig_floor_mean','sig_floor_ratio_mean',
                   'sig_excess_mean','mean_events_per_day','h_drift_reduction']].to_string(index=False))

# Inferential tests
print('\n--- Inferential tests (preliminary, not confirmatory) ---')
conds = {c: params.loc[params['study_condition']==c,'mu_hat'].dropna().values
         for c in ['control','default','reintervention']}

f_stat, p_anova = stats.f_oneway(*[v for v in conds.values() if len(v)>0])
print(f'ANOVA mu_hat: F={f_stat:.3f}, p={p_anova:.4f}')

for c in ['default','reintervention']:
    t_stat, p_val = stats.ttest_ind(conds['control'], conds[c], equal_var=False)
    print(f'Welch t-test control vs {c}: t={t_stat:.3f}, p={p_val:.4f}')

sig_groups = [params.loc[params['study_condition']==c,'sig_hat'].dropna().values
              for c in ['control','default','reintervention']]
f_sig, p_sig = stats.f_oneway(*sig_groups)
print(f'ANOVA sig_hat (raw): F={f_sig:.3f}, p={p_sig:.4f}')

ratio_groups = [params.loc[params['study_condition']==c,'sig_floor_ratio'].dropna().values
                for c in ['control','default','reintervention']]
f_ratio, p_ratio = stats.f_oneway(*ratio_groups)
print(f'ANOVA sig_hat / binomial floor: F={f_ratio:.3f}, p={p_ratio:.4f}')
print(f'Participants with sig_hat below floor: {(params["sig_floor_ratio"] < 1).sum()} / {len(params)}')

ols_groups = [params.loc[params['study_condition']==c,'mu_ols'].dropna().values
              for c in ['control','default','reintervention']]
f_ols, p_ols = stats.f_oneway(*ols_groups)
print(f'ANOVA all-days OLS slope: F={f_ols:.3f}, p={p_ols:.4f}')

In [ ]:
# =============================================================================
# Figure 1: drift robustness and observation-model-adjusted variability
# =============================================================================

COND_ORDER  = ['control','default','reintervention']
COND_COLORS = {'control':'#4C72B0','default':'#DD8452','reintervention':'#55A868'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'DS5 StoP -- Drift Robustness and Sampling-Adjusted Variability\n'
    'X_t is a daily binomial proportion; raw diffusion is not behavioral volatility', fontsize=10)

for ax, col, ylabel, title in [
    (axes[0], 'mu_hat', 'mean diff(X_t)', 'Endpoint drift (telescoping)'),
    (axes[1], 'mu_ols', 'OLS slope per calendar day', 'All-days drift robustness'),
    (axes[2], 'sig_floor_ratio', 'observed SD / binomial floor', 'Sampling-adjusted variability'),
]:
    data = [params.loc[params['study_condition']==c, col].dropna().values for c in COND_ORDER]
    parts = ax.violinplot(data, positions=range(len(COND_ORDER)), showmedians=True, showextrema=False)
    for pc, cond in zip(parts['bodies'], COND_ORDER):
        pc.set_facecolor(COND_COLORS[cond]); pc.set_alpha(0.6)
    parts['cmedians'].set_color('black')
    reference = 1.0 if col == 'sig_floor_ratio' else 0.0
    ax.axhline(reference, color='grey', linestyle='--', lw=0.8, alpha=0.7)
    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels(COND_ORDER)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for i, (vals, cond) in enumerate(zip(data, COND_ORDER)):
        if len(vals):
            ax.text(i, np.nanpercentile(vals, 98),
                    f'n={len(vals)}\nmean={np.mean(vals):.3f}',
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fp = FIGURES / 'fig1_stop_params_by_condition.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()

In [ ]:
# =============================================================================
# Figure 2: mu_hat vs sig_hat scatter -- behavioral state space
# =============================================================================
# Quadrants (divided by mu=0 vertical line):
#   mu > 0, low sigma  = systematic escalation (override rate rising)
#   mu < 0, low sigma  = systematic habituation (override rate declining)
#   high raw sigma      = observed variability plus binomial denominator noise
# PROXY NOTE: describes trajectories in behavioral proxy, not latent psychology.

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle(
    'DS5 StoP -- State Space: mu_hat vs sig_hat\n'
    'WARNING: X_t = behavioral proxy (daily override rate), not latent psychological state',
    fontsize=10)

for cond in COND_ORDER:
    sub = params[params['study_condition'] == cond]
    ax.scatter(sub['mu_hat'], sub['sig_hat'],
               c=COND_COLORS[cond], label=cond, alpha=0.5, s=30, edgecolors='none')

ax.axvline(0, color='grey', linestyle='--', lw=0.8, alpha=0.7)
ax.set_xlabel('mu_hat  (+ = systematic increase in override rate, - = systematic decrease)')
ax.set_ylabel('sig_hat (observed daily-rate variability; includes sampling floor)')
ax.legend(title='Condition')

# Use axis-fraction coordinates for quadrant labels -- robust to any data range
ax.text(0.97, 0.96, 'Noisy escalation\n(mu>0)',        ha='right', va='top',   transform=ax.transAxes, fontsize=8, color='dimgrey', style='italic')
ax.text(0.03, 0.96, 'Noisy habituation\n(mu<0)',        ha='left',  va='top',   transform=ax.transAxes, fontsize=8, color='dimgrey', style='italic')
ax.text(0.97, 0.04, 'Systematic escalation\n(mu>0)',    ha='right', va='bottom',transform=ax.transAxes, fontsize=8, color='dimgrey', style='italic')
ax.text(0.03, 0.04, 'Systematic habituation\n(mu<0)',   ha='left',  va='bottom',transform=ax.transAxes, fontsize=8, color='dimgrey', style='italic')

plt.tight_layout()
fp = FIGURES / 'fig2_stop_state_space.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()

---

## Section 5: Euler-Maruyama Simulation

Forward simulation to qualitatively validate whether the SDE model can reproduce observed trajectory shapes.

In [ ]:
# =============================================================================
# Euler-Maruyama simulator
# =============================================================================

def euler_maruyama(x0, mu, sigma, n_steps, n_trajectories=100, rng=None):
    """
    Simulate dX_t = mu dt + sigma dW_t (Euler-Maruyama, delta_t=1 day).
    X_{n+1} = clip(X_n + mu + sigma*Z_n, 0, 1)
    Returns array of shape (n_trajectories, n_steps+1).
    """
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    traj = np.zeros((n_trajectories, n_steps + 1))
    traj[:, 0] = x0
    for t in range(n_steps):
        noise = rng.standard_normal(n_trajectories)
        traj[:, t+1] = np.clip(traj[:, t] + mu + sigma * noise, 0.0, 1.0)
    return traj


# Select representative participant per condition (closest to condition median)
rng = np.random.default_rng(RNG_SEED)
rep = {}
for cond in COND_ORDER:
    sub = params[(params['study_condition'] == cond) &
                 params['mu_hat'].notna() & params['sig_hat'].notna()].copy()
    if len(sub) == 0:
        continue
    mu_m  = sub['mu_hat'].median()
    sig_m = sub['sig_hat'].median()
    sub['dist'] = np.sqrt(
        ((sub['mu_hat']  - mu_m)  / (sub['mu_hat'].std()  + 1e-9))**2 +
        ((sub['sig_hat'] - sig_m) / (sub['sig_hat'].std() + 1e-9))**2
    )
    rep[cond] = sub.loc[sub['dist'].idxmin(), 'participant_id_canonical']

# sim_records: populated in Figure 3 cell for export
sim_records = []

print('Representative participants (closest to condition median):')
for c, pid in rep.items():
    r = params[params['participant_id_canonical'] == pid].iloc[0]
    print(f'  {c}: {pid} | n_days={r.n_days} | mu={r.mu_hat:.4f} | sig={r.sig_hat:.4f}')

In [ ]:
# =============================================================================
# Figure 3: Observed vs simulated trajectory
# =============================================================================

n_p  = len(rep)
fig, axes = plt.subplots(1, n_p, figsize=(5 * n_p, 4), sharey=True)
if n_p == 1:
    axes = [axes]
fig.suptitle(
    f'DS5 StoP -- Observed vs Euler-Maruyama ({N_TRAJ} trajectories per panel)\n'
    'WARNING: X_t = behavioral proxy (daily override rate)', fontsize=10)

for ax, (cond, pid) in zip(axes, rep.items()):
    obs   = daily_elig[daily_elig['participant_id_canonical'] == pid].sort_values('event_date')
    x_obs = obs['X_t'].values
    n_s   = len(x_obs) - 1
    row   = params[params['participant_id_canonical'] == pid].iloc[0]
    traj  = euler_maruyama(x_obs[0], row.mu_hat, row.sig_hat, n_s, N_TRAJ, rng=rng)
    t_ax  = np.arange(n_s + 1)

    p10 = np.percentile(traj, 10, axis=0)
    p50 = np.median(traj, axis=0)
    p90 = np.percentile(traj, 90, axis=0)

    ax.fill_between(t_ax, p10, p90, alpha=0.25, color=COND_COLORS[cond], label='10-90th pctile sim')
    ax.plot(t_ax, p50, color=COND_COLORS[cond], lw=1.5, ls='--', label='Median sim')
    ax.plot(t_ax, x_obs, color='black', lw=2, label='Observed X_t')
    ax.scatter(t_ax, x_obs, color='black', s=20, zorder=5)
    ax.set_title(f'{cond}\nmu={row.mu_hat:.4f}, sig={row.sig_hat:.4f}', fontsize=10)
    ax.set_xlabel('Days observed')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(0.5, color='grey', ls=':', alpha=0.5)
    if ax is axes[0]:
        ax.set_ylabel('X_t (daily override rate -- behavioral proxy)')
        ax.legend(fontsize=8)

    # Capture simulation summary for export
    for t_i in range(n_s + 1):
        sim_records.append(dict(
            participant_id_canonical=pid,
            study_condition=cond,
            mu_hat=row.mu_hat,
            sig_hat=row.sig_hat,
            t=t_i,
            x_observed=x_obs[t_i] if t_i < len(x_obs) else np.nan,
            sim_p10=p10[t_i],
            sim_p50=p50[t_i],
            sim_p90=p90[t_i],
        ))

plt.tight_layout()
fp = FIGURES / 'fig3_stop_observed_vs_simulated.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()

---

## Section 6: Psychosocial Correlates

Preliminary correlations between SDE parameters and pre-intervention scales. Exploratory only. Restricted to friction-exposed participants (default + reintervention).

In [ ]:
# =============================================================================
# Psychosocial correlates
# =============================================================================

PSYCH = ['participant_id_canonical', 'fomo_mean_pre', 'belong_mean_pre',
         'ostracism_mean_pre', 'digital_wellbeing_pre']
avail = [c for c in PSYCH if c in surv.columns]

params_p = params.merge(surv[avail], on='participant_id_canonical', how='left')
p_fric   = params_p[params_p['study_condition'].isin(['default','reintervention'])].copy()
preds    = [c for c in avail if c != 'participant_id_canonical']

print(f'Friction-exposed N: {len(p_fric)}')
print(f'Predictors available: {preds}\n')

corr_rows = []
for target in ['mu_hat', 'sig_hat']:
    print(f'Correlations with {target}:')
    for pred in preds:
        sub = p_fric[[target, pred]].dropna()
        if len(sub) > 10:
            r, p = stats.pearsonr(sub[target], sub[pred])
            print(f'  {target} ~ {pred:30s}: r={r:+.3f}, p={p:.4f}, n={len(sub)}')
            corr_rows.append(dict(
                target=target, predictor=pred,
                pearson_r=round(r, 4), p_value=round(p, 4), n=len(sub),
                note='friction-exposed conditions only (exploratory)'
            ))
        else:
            print(f'  {target} ~ {pred:30s}: skipped (n={len(sub)} < 10)')
    print()

psych_corr_df = pd.DataFrame(corr_rows)
print(f'Correlation table rows: {len(psych_corr_df)}')

In [ ]:
# =============================================================================
# Figure 4: FoMO vs mu_hat and sig_hat
# =============================================================================

FOMO = 'fomo_mean_pre' if 'fomo_mean_pre' in params_p.columns else None

if FOMO is None:
    print('fomo_mean_pre not available -- skipping Figure 4')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        'DS5 StoP -- Pre-FoMO vs SDE Parameters (friction-exposed conditions)\n'
        'WARNING: SDE parameters derived from behavioral proxy X_t (daily override rate)',
        fontsize=10)

    for ax, ycol, ylabel in [
        (axes[0], 'mu_hat',  'mu_hat (drift)'),
        (axes[1], 'sig_hat', 'sig_hat (diffusion)'),
    ]:
        for cond in ['default', 'reintervention']:
            sub = p_fric[p_fric['study_condition'] == cond][[FOMO, ycol]].dropna()
            ax.scatter(sub[FOMO], sub[ycol], c=COND_COLORS[cond],
                       label=cond, alpha=0.5, s=30, edgecolors='none')
        sub_all = p_fric[[FOMO, ycol]].dropna()
        if len(sub_all) > 5:
            m, b, r, p, _ = stats.linregress(sub_all[FOMO], sub_all[ycol])
            xl = np.linspace(sub_all[FOMO].min(), sub_all[FOMO].max(), 100)
            ax.plot(xl, m * xl + b, color='black', lw=1.5, ls='--',
                    label=f'OLS r={r:.2f}, p={p:.3f}')
        ax.set_xlabel('fomo_mean_pre (pre-intervention FoMO scale mean)')
        ax.set_ylabel(ylabel)
        ax.axhline(0, color='grey', ls=':', alpha=0.5)
        ax.legend(fontsize=9)

    plt.tight_layout()
    fp = FIGURES / 'fig4_stop_fomo_vs_params.png'
    plt.savefig(fp, dpi=150, bbox_inches='tight')
    print(f'Saved: {fp}')
    plt.show()

In [ ]:
# =============================================================================
# Export all model outputs
# =============================================================================

# --- 1. participant_parameters.csv -------------------------------------------
export_params = params[['participant_id_canonical','study_condition',
                         'n_days','n_increments','x_mean','mean_events_per_day',
                         'p_high_days','p_high_days_smoothed',
                         'mu_hat','mu_ols','mu_ols_se',
                         'sig_hat','sig_floor','sig_floor_ratio','sig_excess',
                         'sigma_mu_ratio']].copy()
out1 = REPORTS / 'participant_parameters.csv'
export_params.to_csv(out1, index=False)
print(f'[1] participant_parameters.csv   : {len(export_params):,} rows -> {out1}')

# --- 2. condition_summary.csv ------------------------------------------------
out2 = REPORTS / 'condition_summary.csv'
cond_stats.to_csv(out2, index=False)
print(f'[2] condition_summary.csv         : {len(cond_stats):,} rows -> {out2}')

# --- 3. psychosocial_correlations.csv ----------------------------------------
out3 = REPORTS / 'psychosocial_correlations.csv'
if len(psych_corr_df) > 0:
    psych_corr_df.to_csv(out3, index=False)
    print(f'[3] psychosocial_correlations.csv : {len(psych_corr_df):,} rows -> {out3}')
else:
    print('[3] psychosocial_correlations.csv : no correlations computed (no eligible predictors)')

# --- 4. representative_simulations.csv ---------------------------------------
out4 = REPORTS / 'representative_simulations.csv'
if sim_records:
    sim_df = pd.DataFrame(sim_records)
    sim_df.to_csv(out4, index=False)
    print(f'[4] representative_simulations.csv: {len(sim_df):,} rows -> {out4}')
else:
    print('[4] representative_simulations.csv: no simulation data (rep dict empty)')

# --- 5. model_run_summary.md -------------------------------------------------
n_eligible  = len(params)
n_control   = int((params['study_condition'] == 'control').sum())
n_default   = int((params['study_condition'] == 'default').sum())
n_reint     = int((params['study_condition'] == 'reintervention').sum())
mu_ctrl     = params.loc[params['study_condition']=='control','mu_hat'].mean()
mu_def      = params.loc[params['study_condition']=='default','mu_hat'].mean()
mu_rei      = params.loc[params['study_condition']=='reintervention','mu_hat'].mean()
sig_ctrl    = params.loc[params['study_condition']=='control','sig_hat'].mean()
sig_def     = params.loc[params['study_condition']=='default','sig_hat'].mean()
sig_rei     = params.loc[params['study_condition']=='reintervention','sig_hat'].mean()
h_def       = mu_ctrl - mu_def
h_rei       = mu_ctrl - mu_rei
n_nan_ratio = int(params['sigma_mu_ratio'].isna().sum())

summary_md = f"""# DS5 StoP — Model Run Summary

**Run timestamp:** {MODEL_RUN_TS}
**Notebook:** notebooks/43_ds5_stochastic_processes.ipynb
**Model:** Constant-drift SDE (Stochastic Processes, Stage 4 Step 1)
**Dataset:** DS5 — German Adolescents RCT (N=573 total)

---

## Pipeline parameters

| Parameter | Value |
|---|---|
| MIN_DAYS (minimum observation days) | {MIN_DAYS} |
| N_TRAJ (simulation trajectories) | {N_TRAJ} |
| MU_THRESHOLD (sigma/|mu| NaN guard) | 0.005 |
| RNG_SEED | {RNG_SEED} |

## FLAG decisions in effect

| Flag | Decision |
|---|---|
| FLAG 1 | Simple constant-drift SDE (not OU/mean-reverting) |
| FLAG 2 | Method-of-moments estimators (mean/std of delta_X_n) |
| FLAG 3 | Option A: drift reduction only (h_c = mu_bar_control - mu_bar_c) |
| FLAG 4 | StM Boltzmann bridge deferred |

## Eligibility after MIN_DAYS={MIN_DAYS} threshold

| Condition | N eligible |
|---|---|
| control | {n_control} |
| default | {n_default} |
| reintervention | {n_reint} |
| **Total** | **{n_eligible}** |

## Condition-level SDE parameter means

| Condition | mean mu_hat | mean sig_hat | h_c (drift reduction) |
|---|---|---|---|
| control | {mu_ctrl:.4f} | {sig_ctrl:.4f} | 0.0000 (reference) |
| default | {mu_def:.4f} | {sig_def:.4f} | {h_def:+.4f} |
| reintervention | {mu_rei:.4f} | {sig_rei:.4f} | {h_rei:+.4f} |

## Impulsivity ratio

- sigma/|mu| NaN (|mu| < 0.005): {n_nan_ratio} participants ({100*n_nan_ratio/n_eligible:.1f}%)

## Output files

| File | Description |
|---|---|
| participant_parameters.csv | Per-participant endpoint drift, OLS drift, raw diffusion, binomial floor/ratio/excess, high-day occupancy, n_days, x_mean |
| condition_summary.csv | Condition-level means, medians, std, h_drift_reduction |
| psychosocial_correlations.csv | Pearson r of SDE params vs FoMO/belonging/ostracism (friction-exposed only) |
| representative_simulations.csv | Observed vs E-M simulation (p10/p50/p90) for representative participants |
| figures/fig1_stop_params_by_condition.png | Violin plots of mu_hat and sig_hat by condition |
| figures/fig2_stop_state_space.png | mu_hat vs sig_hat scatter with quadrant labels |
| figures/fig3_stop_observed_vs_simulated.png | Observed trajectory vs E-M simulation envelope |
| figures/fig4_stop_fomo_vs_params.png | Pre-FoMO vs mu_hat and sig_hat scatter |

## Proxy warning

All parameters describe the behavioral proxy X_t = n_opened_day / n_events_day.
X_t approximates behavioral override tendency. It does NOT measure urge, self-control, or any latent psychological state.

## Known limitations

1. Constant-drift SDE (FLAG 1) — misses non-linear and mean-reverting dynamics
2. Method-of-moments (FLAG 2) — noisy for participants with 7-14 day windows
3. Raw sig_hat is not identified as behavioral diffusion: it contains the binomial sampling floor of daily proportions; use sig_floor_ratio or sig_excess
4. Hard clip boundary [0,1] — reflecting boundary or logit SDE would be more principled
5. Stationarity not formally tested
6. Daily aggregation — within-day sequential structure not captured

## Forward connections

- StoP -> DyS (42): sigma->0 limit reduces SDE to ODE
- StoP -> InT (45): H(P_inf) entropy connects sig_hat to behavioral uncertainty
- StoP -> StM (41): Boltzmann bridge deferred (FLAG 4)
- StoP -> Phenotyping (100): (mu_hat, sig_hat, sigma_mu_ratio) feed directly into Step 2
"""

out5 = REPORTS / 'model_run_summary.md'
out5.write_text(summary_md)
print(f'[5] model_run_summary.md          : written -> {out5}')

print(f'\nAll outputs in: {REPORTS}')
print(f'Figures in    : {FIGURES}')

---

## Section 7: Interpretation and Limits

### What the StoP model reveals

| Result | Interpretation | Proxy caveat |
|---|---|---|
| Condition difference in mu_hat | Friction shifts average direction of daily override rate change | mu_hat reflects change in behavioral proxy, not latent urge or habit strength |
| Raw condition difference in sig_hat | Descriptive only; tests of sig_hat/floor determine whether the ordering exceeds count noise | X_t is a binomial daily proportion and raw sig_hat lies near its analytic sampling floor |
| h_c > 0 for friction conditions | Friction reduced average drift toward opening | Effect size depends on condition overlap -- inspect Figure 1 |
| FoMO ~ mu_hat correlation | Higher pre-FoMO associates with drift direction in friction-exposed participants | Pre-FoMO predicts behavioral trajectory, not psychological change |
| sigma/|mu| ratio | Characterizes noise vs drift dominance per participant | High ratio may reflect short windows or genuinely erratic behavior |

### Known limitations

1. **Constant-drift SDE (FLAG 1).** Misses non-linear adaptation. OU process or potential-well SDE are natural extensions.
2. **Method-of-moments (FLAG 2).** Noisy for participants with 7-14 day windows. MLE is a natural upgrade.
3. **Observation model for diffusion.** Raw sig_hat combines behavioral change and binomial daily-denominator noise. The exported floor ratio and excess term are the defensible diagnostics.
4. **Endpoint drift.** mean(diff(X_t)) telescopes to the first/last-day slope; `mu_ols` uses all observed days as a robustness comparison.
5. **Hard clip to [0,1].** A reflecting boundary or logit-scale SDE would be more principled.
6. **Stationarity not tested.** Formally checking for unit roots is recommended for longer-window datasets (DS4).
7. **Daily aggregation.** Within-day sequential patterns not captured.

---

## Section 8: Forward Connections

**StoP -> DyS (notebook 42):** The limit sigma->0 is the ODE dX/dt = mu. Low-sigma participants from Figure 2 are the natural DyS target.

**StoP -> InT (notebook 45):** sig_hat characterizes entropy of the stationary distribution. High-sigma participants have more uncertain behavioral distributions.

**StoP -> StM (FLAG 4, deferred):** For a potential-well SDE, P_inf(x) proportional to exp(-2V(x)/sigma^2). Implementing this requires identifying a non-trivial potential V(x) from data -- deferred to Stage 4 Step 2 or later.

**StoP -> Phenotyping (Stage 4, Step 2):** The (mu_hat, sig_hat, sigma_mu_ratio) triplet feeds directly into behavioral phenotyping (notebook 100). The four quadrants in Figure 2 suggest natural cluster hypotheses: systematic habituators, systematic escalators, noise-dominated. Formal clustering is Step 2 scope.